<a href="https://colab.research.google.com/github/venkata18167/CSA6301---THREAT-INTELLIGENCE-AND-NETWORK-SECURITY/blob/main/32_Inline_IPS_Blocking_Simulator_with_Alert_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
 IDS_RULES = [
    {
        "sid": 2000001,
        "msg": "Possible SQL Injection",
        "proto": "tcp",
        "dst_port": 80,
        "content": "union select"
    },
    {
        "sid": 2000002,
        "msg": "Directory Traversal Attempt",
        "proto": "tcp",
        "dst_port": 80,
        "content": "../../../etc/passwd"
    },
    {
        "sid": 2000003,
        "msg": "Suspicious RDP Brute Force Pattern",
        "proto": "tcp",
        "dst_port": 3389,
        "content": "login_attempt"
    }
]

def scan_packet(packet, rules=IDS_RULES):
    """
    Detect packet signatures based on:
    - Protocol
    - Destination Port
    - Payload Content
    """

    alerts = []

    payload = packet["payload"].lower()

    for rule in rules:
        if (
            packet["proto"] == rule["proto"]
            and packet["dst_port"] == rule["dst_port"]
        ):
            if rule["content"].lower() in payload:
                alerts.append(rule["msg"])

    return alerts
def ips_process(packet, rules, whitelist=None):
    """
    Inline IPS:
    - Allow if no alert.
    - Block if alert and source not whitelisted.
    - Allow with log if source is whitelisted.
    """

    whitelist = whitelist or set()

    alerts = scan_packet(packet, rules)

    if not alerts:
        return {
            "action": "allow",
            "alerts": []
        }

    if packet.get("src_ip") in whitelist:
        return {
            "action": "allow",
            "alerts": alerts,
            "note": "Source whitelisted, alert suppressed from blocking."
        }

    return {
        "action": "block",
        "alerts": alerts
    }
def test_experiment4():
    attack_packet = {
        "proto": "tcp",
        "dst_port": 80,
        "payload": "union select username,password from users",
        "src_ip": "203.0.113.50"
    }
    result = ips_process(attack_packet, IDS_RULES)
    print("Attack Packet:")
    print(result)
    assert result["action"] == "block"
    partner_packet = dict(
        attack_packet,
        src_ip="198.51.100.10"
    )
    result2 = ips_process(
        partner_packet,
        IDS_RULES,
        whitelist={"198.51.100.10"}
    )
    print("\nWhitelisted Partner Packet:")
    print(result2)
    assert result2["action"] == "allow"
    assert "note" in result2
    print("\nAll test cases passed.")
test_experiment4()

Attack Packet:
{'action': 'block', 'alerts': ['Possible SQL Injection']}

Whitelisted Partner Packet:
{'action': 'allow', 'alerts': ['Possible SQL Injection'], 'note': 'Source whitelisted, alert suppressed from blocking.'}

All test cases passed.
